# Scenarios and realism (milestone 4)

This notebook puts terrain, radar physics and a second sensor type into
the simulation:

1. **Terrain.** Moving a radar behind a ridge removes low-altitude
   detections on that bearing: the milestone's done-when check.
2. **Radar-equation Pd.** Detection probability from SNR, falling with
   range and depending on the aircraft's size (its RCS class).
3. **A passive bearing-only sensor.**
4. **The run store.** Runs are stored once, keyed by everything that
   determines them, and reused.

Terrain is real by default: Copernicus GLO-90 tiles for the Front Range,
about 57 MB downloaded once and cached (needs the `terrain` extra:
`uv sync --extra notebook --extra terrain`). Offline, or without the extra,
the notebook falls back to a synthetic ridge between the same two sites.

Traffic is an hour of synthetic flights unless `TRACKING_SIM_RECORDING`
names a capture of your own. SBS captures carry no ADS-B emitter category,
so every aircraft gets the default RCS class in section 4; a capture from
`tracking-sim record aircraft-json` carries it.

| Variable | Meaning |
| --- | --- |
| `TRACKING_SIM_TERRAIN` | `copernicus` (default) or `synthetic` |
| `TRACKING_SIM_SEED` | Run seed (default 42) |
| `TRACKING_SIM_RECORDING` | A capture to use instead: a recording (`.parquet`/`.jsonl`) or SBS text (`.sbs`, `.sbs.gz`) |
| `TRACKING_SIM_WINDOW` | `START,END` in ISO 8601 UTC, to use part of the capture (default: all of it) |
| `TRACKING_SIM_SBS_TZ` | Time zone of SBS timestamps (Dump1090 writes host local time; default UTC) |

In [ ]:
import os
import tempfile
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pymap3d
import yaml

from tracking_sim.cli import parse_time
from tracking_sim.geo import LocalFrame, SensorSite
from tracking_sim.geo.terrain import SyntheticRidge, TerrainHorizon
from tracking_sim.sensors import load_sensor_spec, parse_sensor_spec
from tracking_sim.sensors.detection import pd_swerling, snr_radar_equation
from tracking_sim.sim import ReplayEngine, wilson_interval
from tracking_sim.truth import build_truth
from tracking_sim.truth.sources import open_recording
from tracking_sim.truth.synthetic import ReceiverConfig, generate_traffic

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
TERRAIN = os.environ.get("TRACKING_SIM_TERRAIN", "copernicus")
SEED = int(os.environ.get("TRACKING_SIM_SEED", "42"))
RECORDING = os.environ.get("TRACKING_SIM_RECORDING")
WINDOW = os.environ.get("TRACKING_SIM_WINDOW")
SBS_TZ = os.environ.get("TRACKING_SIM_SBS_TZ", "UTC")

# Colours: the reference data-viz palette's first three categorical slots, plus neutrals.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, GREY, LIGHT = "#52514e", "#8a8984", "#c9c8c2"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

KBJC = (39.9088, -105.1172, 1729.0)  # radar A: Rocky Mountain Metropolitan Airport
SITE_B = (39.92, -105.29)  # radar B: 15 km west, behind the first foothills ridge
frame = LocalFrame(*KBJC)

## 1. Traffic and terrain

An hour of synthetic traffic around KBJC, this time with each flight's ADS-B
emitter category reported (as `aircraft.json` does), which the radar
equation uses below; or your capture, if `TRACKING_SIM_RECORDING` is set.

In [ ]:
if RECORDING:
    window = [parse_time(t) for t in WINDOW.split(",")] if WINDOW else [None, None]
    records = list(open_recording(RECORDING, receiver="capture", tz=SBS_TZ).replay(*window))
    print("capture:", RECORDING)
else:
    traffic = generate_traffic(
        seed=1, receivers=(ReceiverConfig("rx", outlier_count=0, report_category=True),)
    )
    records = traffic.records
truth = build_truth(records, frame)
print(f"{len(truth)} segments; categories: {sorted({s.category for s in truth if s.category})}")

radar_a = load_sensor_spec(ROOT / "scenarios/sensors/radar-1.yaml")
reach = [(KBJC[0], KBJC[1], radar_a.coverage.range_max_m), (*SITE_B, radar_a.coverage.range_max_m)]

terrain = None
if TERRAIN == "copernicus":
    try:
        from tracking_sim.geo.copernicus import area_around, load_area

        terrain = load_area(*area_around(reach))
        source = "copernicus GLO-90"
    except Exception as exc:  # no extra, or offline: fall back
        print(
            f"Copernicus terrain unavailable ({type(exc).__name__}: {exc}); using a synthetic ridge"
        )
if terrain is None:
    # A north-south ridge 6 km east of site B, 900 m above flat ground.
    crest_lat, crest_lon, _ = pymap3d.enu2geodetic(6_000.0, 0.0, 0.0, *SITE_B, 0.0)
    terrain = SyntheticRidge(
        float(crest_lat),
        float(crest_lon),
        azimuth_deg=0.0,
        height_m=900.0,
        width_m=1_500.0,
        length_m=50_000.0,
        base_m=KBJC[2],
    )
    source = "synthetic ridge"
ground_b = float(terrain.height(*SITE_B))
print(
    f"terrain: {source}; site B ground {ground_b:.0f} m, KBJC ground {float(terrain.height(*KBJC[:2])):.0f} m"
)

In [ ]:
lats = np.linspace(39.55, 40.25, 300)
lons = np.linspace(-105.75, -104.65, 400)
grid = terrain.height(lats[:, None], lons[None, :])
fig, ax = plt.subplots(figsize=(9, 5.5))
image = ax.pcolormesh(lons, lats, grid, cmap="Greys", shading="auto")
fig.colorbar(image, ax=ax, label="ground height (m)")
ax.plot(KBJC[1], KBJC[0], "^", color=BLUE, ms=10, label="radar A (KBJC)")
ax.plot(SITE_B[1], SITE_B[0], "^", color=ORANGE, ms=10, label="radar B (behind the ridge)")
ax.set(xlabel="longitude", ylabel="latitude", title=f"Terrain: {source}")
ax.set_aspect(1 / np.cos(np.radians(39.9)))
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()

## 2. Coverage: how low can each radar see?

For each bearing and range, the height above KBJC's ground an aircraft
needs before line of sight clears the terrain (on a 4/3 earth, from a 15 m
mast). Radar A on the plains sees low traffic all round except to the west;
radar B sits behind a ridge, so the plains to its east are in shadow.

In [ ]:
spec_a = radar_a
spec_b = parse_sensor_spec(
    {
        **radar_a.model_dump(),
        "id": "radar-B",
        "position": {"lat": SITE_B[0], "lon": SITE_B[1], "alt_m": ground_b, "mast_m": 15},
    }
)
spec_a = parse_sensor_spec(
    {
        **radar_a.model_dump(),
        "id": "radar-A",
        "position": {**radar_a.position.model_dump(), "alt_m": float(terrain.height(*KBJC[:2]))},
    }
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), subplot_kw={"projection": "polar"})
ranges = np.linspace(1_000, 100_000, 120)
az = np.radians(np.arange(0, 360, 1.0))
for ax, spec, colour in zip(axes, (spec_a, spec_b), (BLUE, ORANGE)):
    site = SensorSite(
        frame, spec.position.lat, spec.position.lon, spec.position.alt_m, spec.position.mast_m
    )
    horizon = TerrainHorizon(site, terrain, 100_000.0, azimuth_step_deg=1.0)
    need = (
        horizon.lowest_visible_height(az[:, None], ranges[None, :]) + spec.position.alt_m - KBJC[2]
    )
    mesh = ax.pcolormesh(
        az,
        ranges / 1e3,
        np.clip(need, 0, 8000).T,
        cmap="viridis",
        vmin=0,
        vmax=8000,
        shading="auto",
    )
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(f"{spec.id}: lowest visible height", color=colour)
fig.colorbar(mesh, ax=axes, label="height above KBJC ground (m)", shrink=0.8)

## 3. Done when: the radar behind the ridge loses low traffic on that bearing

The same radar, same traffic and seed, at site A and at site B. Detections
are coloured by the aircraft's height above KBJC's ground; the bars count
low-altitude detections (below 3,000 m) by bearing from each radar.

In [ ]:
def run(spec):
    quiet = parse_sensor_spec(
        {**spec.model_dump(), "detection": {"pd": 0.9, "false_alarm_density": 0.0}}
    )
    engine = ReplayEngine(truth, [quiet], seed=SEED, terrain=terrain)
    return engine.run()


result_a, result_b = run(spec_a), run(spec_b)
fig = plt.figure(figsize=(17, 5.6), layout="constrained")
axes = fig.subplots(1, 3, gridspec_kw={"width_ratios": [1, 1, 1.1]})
low = {}
for ax, result, spec in zip(axes[:2], (result_a, result_b), (spec_a, spec_b)):
    for seg in truth:
        t = np.arange(seg.t[0], seg.t[-1], 10.0)
        pos, _, ok = seg.evaluate(t)
        ax.plot(pos[ok, 0] / 1e3, pos[ok, 1] / 1e3, color=LIGHT, lw=2, zorder=1)
    opp = result.opportunities_table().to_pandas()
    det = opp[opp["detected"]]
    s = ax.scatter(
        det["east_m"] / 1e3,
        det["north_m"] / 1e3,
        c=np.clip(det["up_m"], 0, 10_000),
        s=3,
        cmap="viridis",
        vmin=0,
        vmax=10_000,
        lw=0,
        zorder=2,
    )
    site = result.radars[spec.id].site.position / 1e3
    ax.plot(*site[:2], "^", color=INK, ms=8, zorder=3)
    gated = result.stats(spec.id).gated.get("terrain", 0)
    ax.set(
        title=f"{spec.id}: {len(det)} detections, {gated} looks hidden",
        xlabel="east (km)",
        xlim=(-115, 115),
        ylim=(-115, 115),
    )
    ax.set_aspect("equal")
    low[spec.id] = opp[opp["in_coverage"] & opp["detected"] & (opp["up_m"] < 3000)]
    low[spec.id + ":hidden"] = opp[(opp["gate"] == "terrain")]
axes[0].set_ylabel("north (km)")
fig.colorbar(s, ax=axes[1], label="height above KBJC ground (m)", shrink=0.8)

bins = np.arange(0, 361, 20)
centres = (bins[:-1] + bins[1:]) / 2
for offset, (name, colour) in enumerate((("radar-A", BLUE), ("radar-B", ORANGE))):
    counts, _ = np.histogram(low[name]["azimuth_deg"], bins=bins)
    axes[2].bar(centres + (offset - 0.5) * 8, counts, width=8, color=colour, label=name)
axes[2].set(
    xlabel="bearing from the radar (deg)",
    ylabel="low-altitude detections",
    title="Below 3,000 m, by bearing",
)
axes[2].set_xticks(range(0, 361, 60))
axes[2].legend(fontsize=8, frameon=False)


def east(rows):
    return rows["azimuth_deg"].between(45, 135)


east_a, east_b = int(east(low["radar-A"]).sum()), int(east(low["radar-B"]).sum())
print(f"low-altitude detections on bearings 45-135 deg: radar A {east_a}, radar B {east_b}")
print(
    f"behind the ridge, radar B loses most low-altitude traffic on those bearings: {east_b < east_a / 2}"
)
hidden = low["radar-B:hidden"]
print(
    f"radar B: {len(hidden)} looks hidden by terrain, {east(hidden).mean():.0%} of them on bearings 45-135 deg, "
    f"median height {hidden['up_m'].median():.0f} m"
)

## 4. Radar-equation Pd and RCS classes

Instead of one Pd, detection now follows the radar equation: a 1 m^2
target at 50 km gives 13 dB, SNR falls with range to the fourth power and
scales with the aircraft's radar cross-section class (from its ADS-B
emitter category). Lines are the model; dots the achieved Pd in this run,
in 10 km bins, with 95% intervals (each aircraft contributes a run of
looks, so a bin is fewer independent samples than it seems).

In [ ]:
rq = parse_sensor_spec(
    {
        **spec_a.model_dump(),
        "id": "radar-eq",
        "detection": {
            "model": "radar_equation",
            "reference_snr_db": 13.0,
            "reference_range_m": 50_000.0,
            "false_alarm_density": 0.0,
        },
    }
)
result = ReplayEngine(truth, [rq], seed=SEED, terrain=terrain).run()
stats = result.stats("radar-eq")
print(stats.summary())

opp = result.opportunities_table().to_pandas()
opp = opp[opp["in_coverage"]]
fig, ax = plt.subplots(figsize=(9, 4.2))
r = np.linspace(5_000, 110_000, 200)
colours = {"light": AQUA, "small": ORANGE, "large": BLUE, "heavy": INK}
for cls, colour in colours.items():
    rcs = getattr(rq.detection.rcs_m2, cls)
    ax.plot(
        r / 1e3,
        pd_swerling(snr_radar_equation(r, rcs, 13.0, 50_000.0), rq.detection.pfa),
        color=colour,
        lw=1.5,
        label=f"{cls} ({rcs:g} m²)",
    )
    rows = opp[opp["rcs_class"] == cls]
    if len(rows) > 50:
        bins = np.arange(0, 120_001, 10_000)
        idx = np.digitize(rows["slant_range_m"], bins)
        for k in np.unique(idx):
            sel = rows[idx == k]
            if len(sel) >= 20:
                k_hit = int(sel["detected"].sum())
                lo, hi = wilson_interval(k_hit, len(sel))
                p = k_hit / len(sel)
                ax.errorbar(
                    sel["slant_range_m"].mean() / 1e3,
                    p,
                    yerr=[[max(p - lo, 0.0)], [max(hi - p, 0.0)]],
                    fmt="o",
                    color=colour,
                    ms=5,
                    capsize=2,
                    lw=1,
                )
ax.set(
    xlabel="slant range (km)",
    ylabel="Pd",
    ylim=(0, 1.02),
    title="Pd by range and RCS class (Swerling 1)",
)
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()

## 5. A passive bearing-only sensor

A staring receiver at KBJC listening to the aircraft: every second it
reports a bearing to each aircraft it hears (Pd falls with range squared,
a one-way link), with 1 degree of noise and some false bearings. It
measures no range, so these detections wait for bearing-only tracking and
fusion in milestone 6.

In [ ]:
esm = parse_sensor_spec(
    {
        "id": "esm-1",
        "type": "passive_bearing",
        "position": spec_a.position.model_dump(),
        "coverage": {"range_max_m": 150_000, "elevation_deg": [-2, 60]},
        "update_period_s": 1.0,
        "noise": {"bearing_sigma_deg": 1.0},
        "detection": {
            "model": "one_way",
            "reference_snr_db": 20.0,
            "reference_range_m": 60_000.0,
            "false_alarm_rate": 0.5,
        },
    }
)
start = result_a.start
passive = ReplayEngine(truth, [esm], seed=SEED, terrain=terrain).run(
    start, start + (result_a.end - start) / 6
)
print(passive.stats("esm-1").summary())
t = passive.detections_table().to_pandas()
t0 = t["timestamp"].min()
fig, ax = plt.subplots(figsize=(10, 4))
targets, clutter = t[t["kind"] == "target"], t[t["kind"] == "clutter"]
ax.scatter(
    (clutter["timestamp"] - t0).dt.total_seconds() / 60,
    clutter["azimuth_deg"],
    s=3,
    color=LIGHT,
    label="false bearings",
)
ax.scatter(
    (targets["timestamp"] - t0).dt.total_seconds() / 60,
    targets["azimuth_deg"],
    s=2,
    color=BLUE,
    label="bearings to aircraft",
)
ax.set(xlabel="minutes", ylabel="bearing (deg)", ylim=(0, 360), title="esm-1: bearings over time")
ax.legend(fontsize=8, frameon=False, markerscale=4)
fig.tight_layout()

## 6. The run store

Scenario runs are stored once under `runs/<name>-<hash>/`, where the hash
covers the resolved scenario, the input data (truth, terrain), the seed and
the code. Running an identical scenario again reuses the stored run; any
change makes a new one. Here, a small scenario in a temporary store.

In [ ]:
from tracking_sim.eval.runstore import RunStore
from tracking_sim.sim.scenario import load_scenario
from tracking_sim.truth.sources.recording import write_records

work = Path(tempfile.mkdtemp())
write_records(work / "truth.parquet", records)
# Seven minutes of the traffic, from five minutes in (or as much as there is).
first, last = min(r.ts for r in records), max(r.ts for r in records)
stop = min(first + timedelta(minutes=12), last)
window = [max(first, stop - timedelta(minutes=7)).isoformat(), stop.isoformat()]
scenario = {
    "name": "kbjc-ridge",
    "truth": {"path": "truth.parquet", "window": window},
    # A synthetic ridge keeps this example quick and offline.
    "terrain": {
        "source": "synthetic",
        "ridge": {"lat": 39.92, "lon": -105.22, "azimuth_deg": 0, "height_m": 900, "base_m": 1729},
    },
    "sensors": [spec_a.model_dump(mode="json")],
    "trackers": [{"id": "ekf", "filter": "ekf"}],
    "seed": SEED,
}
(work / "scenario.yaml").write_text(yaml.safe_dump(scenario))
store = RunStore(work / "runs")
first = store.run(load_scenario(work / "scenario.yaml"))
again = store.run(load_scenario(work / "scenario.yaml"))
print(first.describe())
print(f"\nreused stored run: {again.reused} (same id: {again.id == first.id})")
scenario["seed"] = SEED + 1
(work / "scenario.yaml").write_text(yaml.safe_dump(scenario))
other = store.run(load_scenario(work / "scenario.yaml"))
print(f"another seed: new run {other.id}")
print([r.id for r in store.list()])